# Λ$_s$CDM — 2×2×2×3 Isolation Analysis

Extension of the v1 2×2×2 isolation to include BAOtr as third BAO source.

## Structure

12 fits total: **3 BAO sources × 2 SnIa × 2 CC**, MB always on, prior $z_\dagger \in [1, 3]$.

| Axis | Levels |
|---|---|
| BAO | Akarsu (SDSS-era), DESI DR2, BAOtr |
| SnIa | Pantheon (1048), Pantheon+ (1624) |
| CC | no CC, Moresco 2022 (33 pts) |

## Reuse strategy

8 fits exist from v1 isolation: 4 Akarsu + 4 DR2 (across SnIa/CC axes).
4 fits are new: BAOtr × 2 SnIa × 2 CC.

The notebook automatically detects existing pickle files and re-runs only
what's missing. Configure the `EXISTING_FITS` dict in the next cell to
match your file naming.

## Goal

Quantify the marginal effect of each axis on the Bayesian evidence.
Specifically:

- **BAO upgrade Akarsu → DR2**: expected ~+5.8 ln Z (against ΛsCDM)
- **BAO upgrade Akarsu → BAOtr**: expected to be **negative** (in favor of ΛsCDM)
- **SnIa upgrade Pantheon → Pantheon+**: ~+0.4 (negligible)
- **CC inclusion**: ~+0.2 (negligible)

This complements the triple BAO comparison triangle plots from v2.


## 1. Configuration — Existing Pickle Files

**EDIT THIS CELL** to point to your existing v1 isolation pickle files.

If the pickle path doesn't match your file, the notebook will re-run that
fit (slow). Verify file existence before running the rest of the notebook.

The standard v1 naming convention (from previous sessions) is `iso_*` but
your files may use a different scheme.


In [1]:
import os

EXISTING_FITS = {
}
ISO_NAMES = [
    'iso_v3_akarsu_P_noCC',  'iso_v3_akarsu_P_CC',  'iso_v3_akarsu_PP_noCC',  'iso_v3_akarsu_PP_CC',
    'iso_v3_dr2_P_noCC',     'iso_v3_dr2_P_CC',     'iso_v3_dr2_PP_noCC',     'iso_v3_dr2_PP_CC',
    'iso_v3_baotr_P_noCC',   'iso_v3_baotr_P_CC',   'iso_v3_baotr_PP_noCC',   'iso_v3_baotr_PP_CC',
]

print('Checking existing v3 pickle files:')
print('=' * 70)
import shutil

reused = 0
need_to_run = []
for internal_name in ISO_NAMES:
    local_pkl = f'{internal_name}.pkl'
    if os.path.exists(local_pkl):
        print(f'  ✓ {internal_name}: already at standard name')
        reused += 1
        continue

    src = EXISTING_FITS.get(internal_name)
    if src and os.path.exists(src):
        shutil.copy(src, local_pkl)
        print(f'  ✓ {internal_name}: copied from {src}')
        reused += 1
    else:
        need_to_run.append(internal_name)
        print(f'  ✗ {internal_name}: needs to be run')

print('=' * 70)
print(f'Reused: {reused}/12 fits')
print(f'To run: {len(need_to_run)} fits')
if len(need_to_run) > 0:
    print(f'  ({", ".join(need_to_run)})')
    print(f'Estimated compute: ~30 min per fit')
    print(f'Total: ~{len(need_to_run) * 30} minutes')


Checking existing v3 pickle files:
  ✓ iso_v3_akarsu_P_noCC: already at standard name
  ✓ iso_v3_akarsu_P_CC: already at standard name
  ✓ iso_v3_akarsu_PP_noCC: already at standard name
  ✓ iso_v3_akarsu_PP_CC: already at standard name
  ✓ iso_v3_dr2_P_noCC: already at standard name
  ✓ iso_v3_dr2_P_CC: already at standard name
  ✓ iso_v3_dr2_PP_noCC: already at standard name
  ✓ iso_v3_dr2_PP_CC: already at standard name
  ✓ iso_v3_baotr_P_noCC: already at standard name
  ✓ iso_v3_baotr_P_CC: already at standard name
  ✓ iso_v3_baotr_PP_noCC: already at standard name
  ✓ iso_v3_baotr_PP_CC: already at standard name
Reused: 12/12 fits
To run: 0 fits


## 2. Setup


In [2]:
!pip install nautilus-sampler


In [3]:
import numpy as np
from scipy.integrate import quad
from nautilus import Prior, Sampler
import urllib.request, pandas as pd
import pickle
import warnings
warnings.filterwarnings("ignore")


## 3. Constants and Cosmological Functions


In [4]:
c    = 299792.458
Tcmb = 2.7255
cH0  = 2997.92458

N_LIVE = 1500
NU = 20
N_GRID = 1500

ZT_MAX = 3.0

In [5]:
def E2(z, om, h, w0=-1.0, wa=0.0):
    z = np.asarray(z, dtype=float)
    a = 1.0 / (1.0 + z)
    om_r = 4.18e-5 / h**2
    om_de = 1.0 - om - om_r
    rde = a**(-3.0*(1.0+w0+wa)) * np.exp(-3.0*wa*(1.0-a))
    return om*a**(-3) + om_r*a**(-4) + om_de*rde

def E(z, om, h, w0=-1.0, wa=0.0):
    return np.sqrt(E2(z, om, h, w0, wa))

def H_z(z, om, h, w0=-1.0, wa=0.0):
    return 100.0*h*E(z, om, h, w0, wa)

def D_M(z, om, h, w0=-1.0, wa=0.0):
    result, _ = quad(lambda zp: 1.0/E(zp, om, h, w0, wa), 0.0, z)
    return (cH0/h) * result

def D_H(z, om, h, w0=-1.0, wa=0.0):
    return c / H_z(z, om, h, w0, wa)

def R_baryon(z, obh2):
    return 31500.0*obh2*(Tcmb/2.7)**(-4)/(1.0+z)
def c_s(z, obh2):
    return c/np.sqrt(3.0*(1.0+R_baryon(z, obh2)))
def z_cmb(om, obh2, h):
    om_h2 = om*h**2
    g1 = (0.0783*obh2**(-0.238))/(1.0+39.5*obh2**0.763)
    g2 = 0.560/(1.0+21.1*obh2**1.81)
    return 1048.0*(1.0+0.00124*obh2**(-0.738))*(1.0+g1*om_h2**g2)
def z_drag(om, obh2, h):
    om_h2 = om*h**2
    b1 = 0.313*om_h2**(-0.419)*(1.0+0.607*om_h2**0.674)
    b2 = 0.238*om_h2**0.223
    return (1291.0*om_h2**0.251/(1.0+0.659*om_h2**0.828)
            *(1.0+b1*obh2**b2))


## 4. Smooth $\Lambda_s$CDM Implementation\n

In [6]:
def E2_scdm_smooth(z, om, h, zt, nu=NU):
    z = np.asarray(z, dtype=float)
    a = 1.0/(1.0+z)
    om_r = 4.18e-5/h**2
    om_de = 1.0 - om - om_r
    sign_smooth = np.tanh(nu*(zt - z))/np.tanh(nu*zt)
    return om*a**(-3) + om_r*a**(-4) + om_de*sign_smooth

def E_scdm_smooth(z, om, h, zt, nu=NU):
    return np.sqrt(np.maximum(E2_scdm_smooth(z, om, h, zt, nu), 1e-30))

def H_scdm_smooth(z, om, h, zt, nu=NU):
    return 100.0*h*E_scdm_smooth(z, om, h, zt, nu)

def DL_array_scdm_smooth(z_arr, om, h, zt):
    z_arr = np.asarray(z_arr)
    z_max = z_arr.max() + 0.05
    z_grid = np.concatenate([[0.0],
        np.logspace(np.log10(1e-4), np.log10(z_max), N_GRID-1)])
    z_grid = np.unique(z_grid)
    inv_E = 1.0/E_scdm_smooth(z_grid, om, h, zt)
    chi_grid = np.concatenate([[0.0], np.cumsum(
        0.5*(inv_E[:-1]+inv_E[1:])*np.diff(z_grid))])
    return (1.0+z_arr)*np.interp(z_arr, z_grid, chi_grid)

def D_M_scdm_smooth(z, om, h, zt):
    result, _ = quad(lambda zp: 1.0/E_scdm_smooth(zp, om, h, zt),
                     0.0, float(z))
    return (cH0/h)*result

def D_A_scdm_smooth(z, om, h, zt):
    return D_M_scdm_smooth(z, om, h, zt) / (1.0 + z)

def D_H_scdm_smooth(z, om, h, zt):
    return c/H_scdm_smooth(z, om, h, zt)

def D_V_scdm_smooth(z, om, h, zt):
    dm = D_M_scdm_smooth(z, om, h, zt)
    dh = D_H_scdm_smooth(z, om, h, zt)
    return (z*dm**2*dh)**(1.0/3.0)

def r_s_scdm_smooth(ze_func, om, obh2, h, zt):
    z_epoch = ze_func(om, obh2, h)
    result, _ = quad(lambda z: c_s(z, obh2)/H_scdm_smooth(z, om, h, zt),
                     z_epoch, np.inf)
    return result

def R_shift_scdm_smooth(om, obh2, h, zt):
    z_star = z_cmb(om, obh2, h)
    return np.sqrt(om*(100*h)**2)*D_M_scdm_smooth(z_star, om, h, zt)/c

def l_a_scdm_smooth(om, obh2, h, zt):
    z_star = z_cmb(om, obh2, h)
    return (np.pi*D_M_scdm_smooth(z_star, om, h, zt)
            /r_s_scdm_smooth(z_cmb, om, obh2, h, zt))

def rd_GA(om, h, obh2):
    omh2 = om * h**2
    return 1.0 / (
        0.00785436 * obh2**0.177084 +
        0.00912388 * omh2**0.618711 +
        11.9611 * obh2**2.81343 * omh2**0.784719
    )


## 5. Compressed Planck CMB


In [7]:
data_cmb = np.array([1.74963, 301.80845, 0.02237])
cov_cmb = 1e-8 * np.array([
    [1598.9554,    17112.007,    -36.311179],
    [17112.007,    811208.45,    -494.79813 ],
    [-36.311179,   -494.79813,      2.1242182],
])
invcov_cmb = np.linalg.inv(cov_cmb)

def chi2_cmb_smooth(om, obh2, h, zt):
    theory = np.array([R_shift_scdm_smooth(om, obh2, h, zt),
                       l_a_scdm_smooth(om, obh2, h, zt), obh2])
    v = theory - data_cmb
    return v @ invcov_cmb @ v


## 6. SH0ES $M_B$ Prior


In [8]:
MB_PRIOR_MEAN  = -19.2435
MB_PRIOR_SIGMA = 0.0373

def chi2_MB_prior(M_abs):
    return ((M_abs - MB_PRIOR_MEAN)/MB_PRIOR_SIGMA)**2


## 7. Pantheon (1048 SNe) — Akarsu original choice


In [9]:
url_p_base = ('https://raw.githubusercontent.com/dscolnic/Pantheon/master/')

if not os.path.exists('lcparam_full_long.txt'):
    urllib.request.urlretrieve(url_p_base + 'lcparam_full_long.txt',
                                'lcparam_full_long.txt')
if not os.path.exists('sys_full_long.txt'):
    urllib.request.urlretrieve(url_p_base + 'sys_full_long.txt',
                                'sys_full_long.txt')

with open('lcparam_full_long.txt') as f:
    header = f.readline().lstrip('#').split()

df_p = pd.read_csv('lcparam_full_long.txt', sep=r'\s+', skiprows=1, names=header)
n_p = len(df_p)
z_p_HD  = df_p['zcmb'].values.astype(float)
z_p_hel = df_p['zhel'].values.astype(float)
mb_p    = df_p['mb'].values.astype(float)
dmb_p   = df_p['dmb'].values.astype(float)
z_p_corr = (1.0 + z_p_hel)/(1.0 + z_p_HD)

with open('sys_full_long.txt') as f:
    sys_lines = f.readlines()
n_sys = int(sys_lines[0].strip())
sys_vec = np.array([float(v) for v in sys_lines[1:]])
CovSys_p = sys_vec.reshape(n_p, n_p)
CovStat_p = np.diag(dmb_p**2)
CovTotal_p = CovStat_p + CovSys_p
InvCov_p = np.linalg.inv(CovTotal_p)
print(f'Pantheon: {n_p} SNe')

def chi2_p_smooth(M_abs, om, h, zt):
    DL = z_p_corr * DL_array_scdm_smooth(z_p_HD, om, h, zt)
    mb_th = M_abs + 5.0*np.log10(DL*cH0/h) + 25.0
    delta = mb_p - mb_th
    return delta @ InvCov_p @ delta


Pantheon: 1048 SNe


## 8. Pantheon+ (1624 SNe) — modern


In [10]:
url_pp_base = ('https://raw.githubusercontent.com/PantheonPlusSH0ES/'
               'DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/')

if not os.path.exists('Pantheon+SH0ES.dat'):
    urllib.request.urlretrieve(url_pp_base + 'Pantheon%2BSH0ES.dat',
                                'Pantheon+SH0ES.dat')
if not os.path.exists('Pantheon+SH0ES_STAT+SYS.cov'):
    urllib.request.urlretrieve(url_pp_base + 'Pantheon%2BSH0ES_STAT%2BSYS.cov',
                                'Pantheon+SH0ES_STAT+SYS.cov')

df_pp = pd.read_csv('Pantheon+SH0ES.dat', sep=r'\s+')
n_pp_total = len(df_pp)

with open('Pantheon+SH0ES_STAT+SYS.cov') as f:
    cov_lines = f.readlines()
CovTotal_pp = np.array([float(v) for l in cov_lines[1:] for v in l.split()]
                      ).reshape(n_pp_total, n_pp_total)

mask_pp = df_pp['IS_CALIBRATOR'] == 0
idx_pp = np.where(mask_pp)[0]
z_pp_HD = df_pp.loc[mask_pp, 'zHD'].values.astype(float)
z_pp_hel = df_pp.loc[mask_pp, 'zHEL'].values.astype(float)
mb_pp_corr = df_pp.loc[mask_pp, 'm_b_corr'].values.astype(float)
z_pp_corr = (1.0 + z_pp_hel)/(1.0 + z_pp_HD)
n_pp = mask_pp.sum()
CovCos_pp = CovTotal_pp[np.ix_(idx_pp, idx_pp)]
InvCov_pp = np.linalg.inv(CovCos_pp)
print(f'Pantheon+: {n_pp} SNe')

def chi2_pp_smooth(M_abs, om, h, zt):
    DL = z_pp_corr * DL_array_scdm_smooth(z_pp_HD, om, h, zt)
    mb_th = M_abs + 5.0*np.log10(DL*cH0/h) + 25.0
    delta = mb_pp_corr - mb_th
    return delta @ InvCov_pp @ delta


Pantheon+: 1624 SNe


## 9. Cosmic Chronometers

33 points, block-diagonal covariance — **identical to Part A (`CosmologicalConstraintsFinal`)**, copied verbatim:

- **18 uncorrelated:** Moresco (2024), arXiv:2412.01994, Table 1 (excluding 2 of 3 'a'-flagged duplicates per Pataki+ 2025, A&A)
- **15 correlated:** Moresco et al. (2012, 2015, 2016) BC03 measurements with the full 15×15 syst+stat covariance from Moresco+ 2020 (https://gitlab.com/mmoresco/CCcovariance)
- **Total covariance:** block-diagonal $\mathrm{diag}(\sigma^2)\oplus\mathrm{cov}_{15}$


In [11]:
# ================================================================
# COSMIC CHRONOMETERS — 33 points, full block-diagonal covariance.
# Copied verbatim from CosmologicalConstraintsFinal (Part A, cell 20) so the
# isolation uses the IDENTICAL chronometer dataset AND covariance.
#   18 uncorrelated : Moresco (2024), arXiv:2412.01994, Table 1
#                     (excluding 2 of 3 'a'-flagged duplicates per Pataki+ 2025)
#   15 correlated   : Moresco et al. (2012, 2015, 2016) BC03 measurements with
#                     the full 15x15 syst+stat covariance from Moresco+ 2020
#                     (https://gitlab.com/mmoresco/CCcovariance)
#   Total covariance: block-diagonal  diag(sig^2) (+) cov15
# Requires the local file 'cc_uncorrelated_18pts.csv' (ships with Part A) and,
# on first run, network access to gitlab for the two BC03 files.
# ================================================================
cc_hz_file  = 'HzTable_MM_BC03.dat'
cc_cov_file = 'CovMat_CC_BC03_Hz_15.dat'
cc_unc_file = 'cc_uncorrelated_18pts.csv'
cc_base = 'https://gitlab.com/mmoresco/CCcovariance/-/raw/master/data/'

# Download Moresco-group BC03 files if missing
for f in [cc_hz_file, cc_cov_file]:
    if not os.path.exists(f):
        try:
            print(f'Downloading {f}...')
            urllib.request.urlretrieve(cc_base + f, f)
        except Exception as e:
            print(f'WARNING: {f} download failed: {e}')

# (1) 15 correlated Moresco-group points (preserve file order!)
cc_corr = pd.read_csv(cc_hz_file, comment='#', header=None, sep=',',
                      usecols=[0, 1, 2], names=['z', 'Hz', 'errHz'])
z_corr   = cc_corr['z'].values.astype(float)
H_corr   = cc_corr['Hz'].values.astype(float)
sig_corr = cc_corr['errHz'].values.astype(float)
cov15    = np.loadtxt(cc_cov_file)
assert len(z_corr) == 15, f'Expected 15 correlated points, got {len(z_corr)}'
assert cov15.shape == (15, 15), f'Expected 15x15 covariance, got {cov15.shape}'

# (2) 18 uncorrelated points (Moresco 2024 Table 1, filtered)
cc_unc = pd.read_csv(cc_unc_file, comment='#')
z_unc   = cc_unc['z'].values.astype(float)
H_unc   = cc_unc['Hz'].values.astype(float)
sig_unc = cc_unc['errHz'].values.astype(float)
assert len(z_unc) == 18, f'Expected 18 uncorrelated points, got {len(z_unc)}'

# (3) Concatenate: [18 uncorrelated ; 15 correlated]
z_cc   = np.concatenate([z_unc, z_corr])
H_cc   = np.concatenate([H_unc, H_corr])
sig_cc = np.concatenate([sig_unc, sig_corr])
n_cc   = len(z_cc)

# (4) Block-diagonal covariance: diag(sig^2) (+) cov15
Cov_cc = np.diag(sig_cc**2)
Cov_cc[18:, 18:] = cov15
InvCov_cc = np.linalg.inv(Cov_cc)

print(f'CC: {n_cc} points (18 uncorr + 15 corr), '
      f'z range [{z_cc.min():.2f}, {z_cc.max():.2f}]')
print(f'  Cov shape: {Cov_cc.shape}, cond(Cov) = {np.linalg.cond(Cov_cc):.2e}')

def chi2_cc_smooth(om, h, zt):
    H_th  = np.array([H_scdm_smooth(z, om, h, zt) for z in z_cc])
    delta = H_cc - H_th
    return delta @ InvCov_cc @ delta


CC: 33 points (18 uncorr + 15 corr), z range [0.07, 1.97]
  Cov shape: (33, 33), cond(Cov) = 2.45e+02


## 10. Akarsu BAO (3D, SDSS-era)


In [12]:
bao_iso_akarsu = [
    (0.15,  4.47,  0.17, 0.17),
    (0.85, 18.33,  0.62, 0.57),
]
bao_aniso_akarsu = [
    (0.38, 10.23, 0.17, 25.00, 0.76),
    (0.51, 13.36, 0.21, 22.33, 0.58),
    (0.70, 17.86, 0.33, 19.33, 0.53),
    (1.48, 30.69, 0.80, 13.26, 0.55),
    (2.33, 37.6,  1.9,   8.93, 0.28),
    (2.33, 37.3,  1.7,   9.08, 0.34),
]


def chi2_bao_akarsu_smooth(om, obh2, h, zt):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for z_eff, dv_obs, sig_lo, sig_hi in bao_iso_akarsu:
        dv_th = D_V_scdm_smooth(z_eff, om, h, zt)/rd
        sigma = sig_lo if dv_th < dv_obs else sig_hi
        chi2 += ((dv_obs - dv_th)/sigma)**2
    for z_eff, dm_obs, sig_dm, dh_obs, sig_dh in bao_aniso_akarsu:
        dm_th = D_M_scdm_smooth(z_eff, om, h, zt)/rd
        dh_th = D_H_scdm_smooth(z_eff, om, h, zt)/rd
        chi2 += ((dm_obs-dm_th)/sig_dm)**2 + ((dh_obs-dh_th)/sig_dh)**2
    return chi2


## 11. DESI DR2 BAO (3D, modern)


In [13]:
desi_iso = [
    (0.295, 7.942, 0.075),
]
desi_aniso = [
    (0.510, 13.587, 0.169, 21.863, 0.427, -0.475),
    (0.706, 17.347, 0.180, 19.458, 0.332, -0.423),
    (0.934, 21.574, 0.153, 17.641, 0.193, -0.425),
    (1.321, 27.605, 0.320, 14.178, 0.217, -0.437),
    (1.484, 30.519, 0.758, 12.816, 0.513, -0.489),
    (2.330, 38.988, 0.531,  8.632, 0.101, -0.431),
]


def chi2_bao_desi_smooth(om, obh2, h, zt):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for z_eff, dv_obs, sigma in desi_iso:
        chi2 += ((dv_obs - D_V_scdm_smooth(z_eff, om, h, zt)/rd)/sigma)**2
    for z_eff, dm_obs, sig_dm, dh_obs, sig_dh, r_corr in desi_aniso:
        dm_th = D_M_scdm_smooth(z_eff, om, h, zt)/rd
        dh_th = D_H_scdm_smooth(z_eff, om, h, zt)/rd
        cov = np.array([[sig_dm**2, r_corr*sig_dm*sig_dh],
                        [r_corr*sig_dm*sig_dh, sig_dh**2]])
        delta = np.array([dm_obs - dm_th, dh_obs - dh_th])
        chi2 += delta @ np.linalg.inv(cov) @ delta
    return chi2


## 12. BAOtr — Transversal BAO (Nunes et al. 2002.09293)


In [14]:
baotr_data = np.array([
    (0.110, 19.80, 3.26),
    (0.235,  9.06, 0.23),
    (0.365,  6.33, 0.22),
    (0.450,  4.77, 0.17),
    (0.470,  5.02, 0.25),
    (0.490,  4.99, 0.21),
    (0.510,  4.81, 0.17),
    (0.530,  4.29, 0.30),
    (0.550,  4.25, 0.25),
    (0.570,  4.59, 0.36),
    (0.590,  4.39, 0.33),
    (0.610,  3.85, 0.31),
    (0.630,  3.90, 0.43),
    (0.650,  3.55, 0.16),
    (2.225,  1.77, 0.31),
])
z_baotr     = baotr_data[:, 0]
theta_baotr = baotr_data[:, 1]
sig_baotr   = baotr_data[:, 2]


def chi2_baotr_smooth(om, obh2, h, zt):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for i, z in enumerate(z_baotr):
        DA = D_A_scdm_smooth(z, om, h, zt)
        theta_th = rd / ((1.0 + z) * DA) * (180.0 / np.pi)
        chi2 += ((theta_baotr[i] - theta_th) / sig_baotr[i])**2
    return chi2


## 13. Isolation Configuration Builder

Each isolation fit is identified by a tuple (BAO source, SnIa, CC).
MB prior is always applied.


In [15]:
def chi2_isolation(bao_source, snia, cc, om, h, obh2, zt, M):
    """Build chi-squared for isolation fit with MB always on."""
    chi2 = chi2_cmb_smooth(om, obh2, h, zt)

    # SnIa
    if snia == 'P':
        chi2 += chi2_p_smooth(M, om, h, zt)
    elif snia == 'PP':
        chi2 += chi2_pp_smooth(M, om, h, zt)
    else:
        raise ValueError(f"Unknown SnIa: {snia}")

    # BAO
    if bao_source == 'akarsu':
        chi2 += chi2_bao_akarsu_smooth(om, obh2, h, zt)
    elif bao_source == 'dr2':
        chi2 += chi2_bao_desi_smooth(om, obh2, h, zt)
    elif bao_source == 'baotr':
        chi2 += chi2_baotr_smooth(om, obh2, h, zt)
    else:
        raise ValueError(f"Unknown BAO: {bao_source}")

    # CC
    if cc == 'CC':
        chi2 += chi2_cc_smooth(om, h, zt)
    elif cc != 'noCC':
        raise ValueError(f"Unknown CC: {cc}")

    # MB prior always on for isolation
    chi2 += chi2_MB_prior(M)

    return chi2


def make_prior_lsCDM():
    p = Prior()
    p.add_parameter('om',   dist=(0.10, 0.50))
    p.add_parameter('h',    dist=(0.55, 0.85))
    p.add_parameter('obh2', dist=(0.020, 0.025))
    p.add_parameter('zt',   dist=(1.0, ZT_MAX))
    p.add_parameter('M',    dist=(-20.0, -18.5))
    return p


def parse_iso_name(name):
    parts = name.split('_')
    if parts[-1] == 'lcdm':
        parts = parts[:-1]
    parts = parts[1:]
    if parts and parts[0].startswith('v') and parts[0][1:].isdigit():
        parts = parts[1:]
    return parts[0], parts[1], parts[2]


## 14. Run-or-Load Helper


In [16]:
def logZ_with_err(sampler):
    log_z = float(sampler.log_z)
    try:
        _, log_w, _ = sampler.posterior()
        w = np.exp(log_w - log_w.max())
        w_norm = w/w.sum()
        n_eff = 1.0/np.sum(w_norm**2)
        H = np.sum(w_norm*(log_w - np.log(np.sum(np.exp(log_w))))**2)
        return log_z, float(np.sqrt(H/n_eff))
    except Exception:
        return log_z, float('nan')


def run_or_load_iso(name):
    fname = f'{name}.pkl'
    if os.path.exists(fname):
        print(f'  [{name}] loaded from cache')
        with open(fname, 'rb') as f:
            return pickle.load(f)
    
    print(f'  [{name}] running...')
    bao, snia, cc = parse_iso_name(name)
    prior = make_prior_lsCDM()
    
    chi2_fn = lambda p: chi2_isolation(bao, snia, cc,
        p['om'], p['h'], p['obh2'], p['zt'], p['M'])
    loglike = lambda p: -0.5 * chi2_fn(p)
    
    sampler = Sampler(prior, loglike, n_live=N_LIVE)
    sampler.run(verbose=True)
    pts, lw, ll = sampler.posterior()
    w = np.exp(lw - lw.max()); w /= w.sum()
    chi2_min = -2 * ll.max()
    log_z, log_z_err = logZ_with_err(sampler)
    
    result = {
        'name': name, 'points': pts, 'weights': w, 'log_like': ll,
        'log_z': log_z, 'log_z_err': log_z_err, 'chi2_min': chi2_min,
        'config': {'bao': bao, 'snia': snia, 'cc': cc},
    }
    with open(fname, 'wb') as f:
        pickle.dump(result, f)
    print(f'  [{name}] saved → ln Z = {log_z:.2f} ± {log_z_err:.2f}, '
          f'χ²_min = {chi2_min:.2f}')
    return result


## 15. Run All 12 Isolation Fits

Auto-resumable. Reused fits load instantly. New fits take ~30 min each.

Full matrix (3 BAO × 2 SnIa × 2 CC):

| | Akarsu | DR2 | BAOtr |
|---|---|---|---|
| **P, no CC** | reuse | reuse | RUN |
| **P, CC** | reuse | reuse | RUN |
| **PP, no CC** | reuse | reuse | reuse (from v2) |
| **PP, CC** | reuse | reuse | RUN |

Total new BAOtr fits: 3 (since PP+noCC reuses tri2_baotr_pan_bao_mb_p3.pkl).


In [17]:
results = {}
for name in ISO_NAMES:
    r = run_or_load_iso(name)
    results[name] = r

print()
print('=' * 70)
print('  ALL 12 ISOLATION FITS COMPLETE')
print('=' * 70)


  [iso_v3_akarsu_P_noCC] loaded from cache
  [iso_v3_akarsu_P_CC] loaded from cache
  [iso_v3_akarsu_PP_noCC] loaded from cache
  [iso_v3_akarsu_PP_CC] loaded from cache
  [iso_v3_dr2_P_noCC] loaded from cache
  [iso_v3_dr2_P_CC] loaded from cache
  [iso_v3_dr2_PP_noCC] loaded from cache
  [iso_v3_dr2_PP_CC] loaded from cache
  [iso_v3_baotr_P_noCC] loaded from cache
  [iso_v3_baotr_P_CC] loaded from cache
  [iso_v3_baotr_PP_noCC] loaded from cache
  [iso_v3_baotr_PP_CC] loaded from cache

  ALL 12 ISOLATION FITS COMPLETE


## 16. Marginal Effects Analysis

For each axis (BAO, SnIa, CC), compute the marginal effect on ln Z by
averaging over the other two axes.

The marginal effect of axis X going from level $A$ to level $B$ is:

$$\Delta\ln Z_X = \langle \ln Z(B, ...) - \ln Z(A, ...) \rangle$$

where the average is over all combinations of the other axes.


In [18]:
def get_lnz(bao, snia, cc):
    name = f'iso_v3_{bao}_{snia}_{cc}'
    return results[name]['log_z']


# BAO axis: Akarsu → DR2 (averaged over SnIa, CC)
delta_akarsu_to_dr2 = np.mean([
    get_lnz('dr2', s, c) - get_lnz('akarsu', s, c)
    for s in ['P', 'PP'] for c in ['noCC', 'CC']
])

# BAO axis: Akarsu → BAOtr (averaged over SnIa, CC)
delta_akarsu_to_baotr = np.mean([
    get_lnz('baotr', s, c) - get_lnz('akarsu', s, c)
    for s in ['P', 'PP'] for c in ['noCC', 'CC']
])

# SnIa axis: P → PP (averaged over BAO, CC)
delta_p_to_pp = np.mean([
    get_lnz(b, 'PP', c) - get_lnz(b, 'P', c)
    for b in ['akarsu', 'dr2', 'baotr'] for c in ['noCC', 'CC']
])

# CC axis: noCC → CC (averaged over BAO, SnIa)
delta_noCC_to_CC = np.mean([
    get_lnz(b, s, 'CC') - get_lnz(b, s, 'noCC')
    for b in ['akarsu', 'dr2', 'baotr'] for s in ['P', 'PP']
])


print('=' * 75)
print('  MARGINAL EFFECTS ON THE Λ_sCDM EVIDENCE ALONE')
print('  (diagnostic only — NOT the model-preference Bayes factor)')
print('=' * 75)
print()
print(f'  BAO axis: Akarsu → DR2:   Δ lnZ(Λ_sCDM) = {delta_akarsu_to_dr2:+.3f}')
print(f'  BAO axis: Akarsu → BAOtr: Δ lnZ(Λ_sCDM) = {delta_akarsu_to_baotr:+.3f}')
print(f'  SnIa axis: P → PP:        Δ lnZ(Λ_sCDM) = {delta_p_to_pp:+.3f}')
print(f'  CC axis: noCC → CC:       Δ lnZ(Λ_sCDM) = {delta_noCC_to_CC:+.3f}')
print()
print('=' * 75)
print()
print('NOTE: these are shifts in lnZ(Λ_sCDM) by itself. They also move when the')
print('      ΛCDM baseline moves, so on their own they do NOT tell you whether a')
print('      data swap favors or disfavors Λ_sCDM. Model PREFERENCE is set by the')
print('      Bayes factor  ΔlnZ = lnZ(Λ_sCDM) − lnZ(ΛCDM)  — see the')
print("      'MARGINAL EFFECTS ON BAYES FACTOR' cell below (the canonical result).")

  MARGINAL EFFECTS ON THE Λ_sCDM EVIDENCE ALONE
  (diagnostic only — NOT the model-preference Bayes factor)

  BAO axis: Akarsu → DR2:   Δ lnZ(Λ_sCDM) = -0.086
  BAO axis: Akarsu → BAOtr: Δ lnZ(Λ_sCDM) = +1.377
  SnIa axis: P → PP:        Δ lnZ(Λ_sCDM) = -219.083
  CC axis: noCC → CC:       Δ lnZ(Λ_sCDM) = -7.363


NOTE: these are shifts in lnZ(Λ_sCDM) by itself. They also move when the
      ΛCDM baseline moves, so on their own they do NOT tell you whether a
      data swap favors or disfavors Λ_sCDM. Model PREFERENCE is set by the
      Bayes factor  ΔlnZ = lnZ(Λ_sCDM) − lnZ(ΛCDM)  — see the
      'MARGINAL EFFECTS ON BAYES FACTOR' cell below (the canonical result).


## 17. Full Isolation Table for Paper

12-row table with all combinations and their ln Z values.
Format suitable for LaTeX export.


In [19]:
print('=' * 80)
print('  FULL ISOLATION TABLE — 12 fits')
print('=' * 80)
print()
print(f'  {"BAO":<8} {"SnIa":<6} {"CC":<6} {"ln Z":<14} {"χ²_min":<10}  {"Notes"}')
print('-' * 80)

# Sort by BAO source for readability
bao_order = ['akarsu', 'dr2', 'baotr']
ordered_names = [
    f'iso_v3_{b}_{s}_{c}'
    for b in bao_order for s in ['P', 'PP'] for c in ['noCC', 'CC']
]

for name in ordered_names:
    r = results[name]
    bao, snia, cc = parse_iso_name(name)
    lnz = r['log_z']
    err = r['log_z_err']
    chi2 = r['chi2_min']
    
    notes = ''
    if (bao == 'akarsu' and snia == 'P' and cc == 'noCC'):
        notes = '← Akarsu original'
    elif (bao == 'baotr' and snia == 'PP' and cc == 'noCC'):
        notes = '← reused from v2'
    elif (bao == 'dr2' and snia == 'PP' and cc == 'CC'):
        notes = '← modern flagship'
    
    print(f'  {bao:<8} {snia:<6} {cc:<6} '
          f'{lnz:>8.2f} ± {err:.2f} {chi2:>8.2f}  {notes}')

print('=' * 80)
print()
print('Reference for comparison (v1 isolation, 2×2×2 only):')
print(f'  Δln Z (Akarsu → DR2)   = +5.8 ± 0.12 (88% of total marginal effect)')
print(f'  Δln Z (P → PP)         = +0.4 ± 0.08')
print(f'  Δln Z (no CC → CC)     = +0.2 ± 0.05')
print()
print('New (this notebook, 2×2×2×3):')
print(f'  Δln Z (Akarsu → DR2)   = {delta_akarsu_to_dr2:+.3f}')
print(f'  Δln Z (Akarsu → BAOtr) = {delta_akarsu_to_baotr:+.3f}')
print(f'  Δln Z (P → PP)         = {delta_p_to_pp:+.3f}')
print(f'  Δln Z (no CC → CC)     = {delta_noCC_to_CC:+.3f}')


  FULL ISOLATION TABLE — 12 fits

  BAO      SnIa   CC     ln Z           χ²_min      Notes
--------------------------------------------------------------------------------
  akarsu   P      noCC    -543.68 ± 0.08  1050.10  ← Akarsu original
  akarsu   P      CC      -550.98 ± 0.08  1064.71  
  akarsu   PP     noCC    -761.81 ± 0.08  1486.02  
  akarsu   PP     CC      -769.08 ± 0.08  1500.63  
  dr2      P      noCC    -543.75 ± 0.09  1049.94  
  dr2      P      CC      -551.06 ± 0.09  1064.59  
  dr2      PP     noCC    -761.89 ± 0.09  1485.77  
  dr2      PP     CC      -769.20 ± 0.09  1500.34  ← modern flagship
  baotr    P      noCC    -540.76 ± 0.08  1043.91  
  baotr    P      CC      -548.27 ± 0.08  1057.94  
  baotr    PP     noCC    -761.77 ± 0.09  1485.79  ← reused from v2
  baotr    PP     CC      -769.24 ± 0.08  1500.17  

Reference for comparison (v1 isolation, 2×2×2 only):
  Δln Z (Akarsu → DR2)   = +5.8 ± 0.12 (88% of total marginal effect)
  Δln Z (P → PP)         = +0

## 18. LaTeX Table for Paper


In [20]:
header = r"""\begin{table}
\caption{$2\times 2\times 2 \times 3$ isolation analysis: $\ln Z$ for
each combination of BAO compilation, SnIa sample, and Cosmic
Chronometer choice. The $M_B$ prior from SH0ES is applied throughout.
The $z_\dagger$ prior is $[1, 3]$.}
\centering
\begin{tabular}{lll|cc}
\toprule
BAO & SnIa & CC & $\ln Z$ & $\chi^2_{\min}$ \\
\midrule"""

footer = r"""\bottomrule
\end{tabular}
\end{table}"""

print(header)

bao_order = ['akarsu', 'dr2', 'baotr']
ordered_names = [
    f'iso_v3_{b}_{s}_{c}'
    for b in bao_order for s in ['P', 'PP'] for c in ['noCC', 'CC']
]
bao_label = {'akarsu': 'Akarsu (SDSS)', 'dr2': 'DESI DR2', 'baotr': 'BAOtr'}
snia_label = {'P': 'Pantheon', 'PP': 'Pantheon+'}
cc_label = {'noCC': 'none', 'CC': 'Moresco 2022'}

for name in ordered_names:
    r = results[name]
    bao, snia, cc = parse_iso_name(name)
    lnz = r['log_z']
    err = r['log_z_err']
    chi2 = r['chi2_min']
    line = (f'{bao_label[bao]} & {snia_label[snia]} & {cc_label[cc]} '
            f'& ${lnz:.2f} \\pm {err:.2f}$ & ${chi2:.2f}$ \\\\')
    print(line)

print(footer)


\begin{table}
\caption{$2\times 2\times 2 \times 3$ isolation analysis: $\ln Z$ for
each combination of BAO compilation, SnIa sample, and Cosmic
Chronometer choice. The $M_B$ prior from SH0ES is applied throughout.
The $z_\dagger$ prior is $[1, 3]$.}
\centering
\begin{tabular}{lll|cc}
\toprule
BAO & SnIa & CC & $\ln Z$ & $\chi^2_{\min}$ \\
\midrule
Akarsu (SDSS) & Pantheon & none & $-543.68 \pm 0.08$ & $1050.10$ \\
Akarsu (SDSS) & Pantheon & Moresco 2022 & $-550.98 \pm 0.08$ & $1064.71$ \\
Akarsu (SDSS) & Pantheon+ & none & $-761.81 \pm 0.08$ & $1486.02$ \\
Akarsu (SDSS) & Pantheon+ & Moresco 2022 & $-769.08 \pm 0.08$ & $1500.63$ \\
DESI DR2 & Pantheon & none & $-543.75 \pm 0.09$ & $1049.94$ \\
DESI DR2 & Pantheon & Moresco 2022 & $-551.06 \pm 0.09$ & $1064.59$ \\
DESI DR2 & Pantheon+ & none & $-761.89 \pm 0.09$ & $1485.77$ \\
DESI DR2 & Pantheon+ & Moresco 2022 & $-769.20 \pm 0.09$ & $1500.34$ \\
BAOtr & Pantheon & none & $-540.76 \pm 0.08$ & $1043.91$ \\
BAOtr & Pantheon & Moresco 20

---

# Part 2: ΛCDM Fits and Bayes Factor Analysis

The above cells produce LsCDM fits and raw ln Z values. To compute
**Bayes factors** (the physically meaningful quantity for model
comparison), we need ΛCDM fits with the **same data combinations**.

## 19. ΛCDM Cosmological Functions

ΛCDM is the special case of LsCDM with $z_\dagger \to \infty$, i.e.,
no sign-switching. We re-implement standard ΛCDM functions directly
to avoid relying on the smooth limit (which would require very large
$z_\dagger$ in the prior).

The parameter space drops from 5 to 4: $(\Omega_m, h, \omega_b h^2, M_B)$.


In [21]:
def H_lcdm(z, om, h):
    """Hubble parameter in ΛCDM."""
    z = np.asarray(z, dtype=float)
    a = 1.0 / (1.0 + z)
    om_r = 4.18e-5 / h**2
    om_de = 1.0 - om - om_r
    E2 = om*a**(-3) + om_r*a**(-4) + om_de
    return 100.0*h*np.sqrt(E2)


def E_lcdm(z, om, h):
    return H_lcdm(z, om, h) / (100.0*h)


def DL_array_lcdm(z_arr, om, h):
    z_arr = np.asarray(z_arr)
    z_max = z_arr.max() + 0.05
    z_grid = np.concatenate([[0.0],
        np.logspace(np.log10(1e-4), np.log10(z_max), N_GRID-1)])
    z_grid = np.unique(z_grid)
    inv_E = 1.0 / E_lcdm(z_grid, om, h)
    chi_grid = np.concatenate([[0.0], np.cumsum(
        0.5*(inv_E[:-1]+inv_E[1:])*np.diff(z_grid))])
    return (1.0+z_arr)*np.interp(z_arr, z_grid, chi_grid)


def D_M_lcdm(z, om, h):
    result, _ = quad(lambda zp: 1.0/E_lcdm(zp, om, h), 0.0, float(z))
    return (cH0/h)*result


def D_A_lcdm(z, om, h):
    return D_M_lcdm(z, om, h) / (1.0 + z)


def D_H_lcdm(z, om, h):
    return c / H_lcdm(z, om, h)


def D_V_lcdm(z, om, h):
    dm = D_M_lcdm(z, om, h)
    dh = D_H_lcdm(z, om, h)
    return (z*dm**2*dh)**(1.0/3.0)


def r_s_lcdm(ze_func, om, obh2, h):
    z_epoch = ze_func(om, obh2, h)
    result, _ = quad(lambda z: c_s(z, obh2)/H_lcdm(z, om, h),
                     z_epoch, np.inf)
    return result


def R_shift_lcdm(om, obh2, h):
    z_star = z_cmb(om, obh2, h)
    return np.sqrt(om*(100*h)**2)*D_M_lcdm(z_star, om, h)/c


def l_a_lcdm(om, obh2, h):
    z_star = z_cmb(om, obh2, h)
    return np.pi*D_M_lcdm(z_star, om, h)/r_s_lcdm(z_cmb, om, obh2, h)


## 20. ΛCDM Likelihoods


In [22]:
def chi2_cmb_lcdm(om, obh2, h):
    theory = np.array([R_shift_lcdm(om, obh2, h),
                       l_a_lcdm(om, obh2, h), obh2])
    v = theory - data_cmb
    return v @ invcov_cmb @ v


def chi2_p_lcdm(M_abs, om, h):
    DL = z_p_corr * DL_array_lcdm(z_p_HD, om, h)
    mb_th = M_abs + 5.0*np.log10(DL*cH0/h) + 25.0
    delta = mb_p - mb_th
    return delta @ InvCov_p @ delta


def chi2_pp_lcdm(M_abs, om, h):
    DL = z_pp_corr * DL_array_lcdm(z_pp_HD, om, h)
    mb_th = M_abs + 5.0*np.log10(DL*cH0/h) + 25.0
    delta = mb_pp_corr - mb_th
    return delta @ InvCov_pp @ delta


def chi2_cc_lcdm(om, h):
    H_th  = np.array([H_lcdm(z, om, h) for z in z_cc])
    delta = H_cc - H_th
    return delta @ InvCov_cc @ delta


def chi2_bao_akarsu_lcdm(om, obh2, h):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for z_eff, dv_obs, sig_lo, sig_hi in bao_iso_akarsu:
        dv_th = D_V_lcdm(z_eff, om, h)/rd
        sigma = sig_lo if dv_th < dv_obs else sig_hi
        chi2 += ((dv_obs - dv_th)/sigma)**2
    for z_eff, dm_obs, sig_dm, dh_obs, sig_dh in bao_aniso_akarsu:
        dm_th = D_M_lcdm(z_eff, om, h)/rd
        dh_th = D_H_lcdm(z_eff, om, h)/rd
        chi2 += ((dm_obs-dm_th)/sig_dm)**2 + ((dh_obs-dh_th)/sig_dh)**2
    return chi2


def chi2_bao_desi_lcdm(om, obh2, h):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for z_eff, dv_obs, sigma in desi_iso:
        chi2 += ((dv_obs - D_V_lcdm(z_eff, om, h)/rd)/sigma)**2
    for z_eff, dm_obs, sig_dm, dh_obs, sig_dh, r_corr in desi_aniso:
        dm_th = D_M_lcdm(z_eff, om, h)/rd
        dh_th = D_H_lcdm(z_eff, om, h)/rd
        cov = np.array([[sig_dm**2, r_corr*sig_dm*sig_dh],
                        [r_corr*sig_dm*sig_dh, sig_dh**2]])
        delta = np.array([dm_obs - dm_th, dh_obs - dh_th])
        chi2 += delta @ np.linalg.inv(cov) @ delta
    return chi2


def chi2_baotr_lcdm(om, obh2, h):
    rd = rd_GA(om, h, obh2)
    chi2 = 0.0
    for i, z in enumerate(z_baotr):
        DA = D_A_lcdm(z, om, h)
        theta_th = rd / ((1.0 + z) * DA) * (180.0 / np.pi)
        chi2 += ((theta_baotr[i] - theta_th) / sig_baotr[i])**2
    return chi2


## 21. ΛCDM Configuration and Run-or-Load


In [23]:
def chi2_isolation_lcdm(bao_source, snia, cc, om, h, obh2, M):
    """ΛCDM chi-squared, MB always on (parallel to LsCDM dispatcher)."""
    chi2 = chi2_cmb_lcdm(om, obh2, h)

    if snia == 'P':
        chi2 += chi2_p_lcdm(M, om, h)
    elif snia == 'PP':
        chi2 += chi2_pp_lcdm(M, om, h)
    else:
        raise ValueError(f"Unknown SnIa: {snia}")

    if bao_source == 'akarsu':
        chi2 += chi2_bao_akarsu_lcdm(om, obh2, h)
    elif bao_source == 'dr2':
        chi2 += chi2_bao_desi_lcdm(om, obh2, h)
    elif bao_source == 'baotr':
        chi2 += chi2_baotr_lcdm(om, obh2, h)
    else:
        raise ValueError(f"Unknown BAO: {bao_source}")

    if cc == 'CC':
        chi2 += chi2_cc_lcdm(om, h)
    elif cc != 'noCC':
        raise ValueError(f"Unknown CC: {cc}")

    chi2 += chi2_MB_prior(M)
    return chi2


def make_prior_lcdm():
    """ΛCDM prior: 4 parameters (no z†)."""
    p = Prior()
    p.add_parameter('om',   dist=(0.10, 0.50))
    p.add_parameter('h',    dist=(0.55, 0.85))
    p.add_parameter('obh2', dist=(0.020, 0.025))
    p.add_parameter('M',    dist=(-20.0, -18.5))
    return p


def run_or_load_iso_lcdm(name):
    """Same as run_or_load_iso but for ΛCDM (4 parameters)."""
    fname = f'{name}.pkl'
    if os.path.exists(fname):
        print(f'  [{name}] loaded from cache')
        with open(fname, 'rb') as f:
            return pickle.load(f)

    print(f'  [{name}] running...')
    bao, snia, cc = parse_iso_name(name)
    prior = make_prior_lcdm()

    chi2_fn = lambda p: chi2_isolation_lcdm(bao, snia, cc,
        p['om'], p['h'], p['obh2'], p['M'])
    loglike = lambda p: -0.5 * chi2_fn(p)

    sampler = Sampler(prior, loglike, n_live=N_LIVE)
    sampler.run(verbose=True)
    pts, lw, ll = sampler.posterior()
    w = np.exp(lw - lw.max()); w /= w.sum()
    chi2_min = -2 * ll.max()
    log_z, log_z_err = logZ_with_err(sampler)

    result = {
        'name': name, 'points': pts, 'weights': w, 'log_like': ll,
        'log_z': log_z, 'log_z_err': log_z_err, 'chi2_min': chi2_min,
        'config': {'bao': bao, 'snia': snia, 'cc': cc, 'model': 'lcdm'},
    }
    with open(fname, 'wb') as f:
        pickle.dump(result, f)
    print(f'  [{name}] saved → ln Z = {log_z:.2f} ± {log_z_err:.2f}, '
          f'χ²_min = {chi2_min:.2f}')
    return result


## 22. ΛCDM Existing Pickles + Run All 12

You have 8 ΛCDM fits from v1 with binary-encoded names:
- `iso_<BAO><SnIa><CC>_lcdm.pkl` where 0=Akarsu/Pantheon/noCC, 1=DR2/Pantheon+/CC

The notebook copies these to standardized names matching our LsCDM
naming convention (`iso_<bao>_<snia>_<cc>_lcdm.pkl`).

4 new BAOtr ΛCDM fits run from scratch (~2 hours total).


In [24]:
# Mapping: v1 binary-encoded names → our standardized v3 names
# All previous LCDM fits used EH98 — cannot reuse them.
EXISTING_LCDM_FITS = {
    # Previous fits with EH98 cannot be reused.
}

ISO_NAMES_LCDM = [n + '_lcdm' for n in ISO_NAMES]

print('Checking ΛCDM v3 pickle files:')
print('=' * 70)
reused_lcdm = 0
need_to_run_lcdm = []
for internal_name in ISO_NAMES_LCDM:
    local_pkl = f'{internal_name}.pkl'
    if os.path.exists(local_pkl):
        print(f'  ✓ {internal_name}: already at standard name')
        reused_lcdm += 1
        continue

    src = EXISTING_LCDM_FITS.get(internal_name)
    if src and os.path.exists(src):
        shutil.copy(src, local_pkl)
        print(f'  ✓ {internal_name}: copied from {src}')
        reused_lcdm += 1
    else:
        need_to_run_lcdm.append(internal_name)
        print(f'  ✗ {internal_name}: needs to be run')

print('=' * 70)
print(f'Reused ΛCDM: {reused_lcdm}/12 fits')
print(f'To run ΛCDM: {len(need_to_run_lcdm)} fits')
if len(need_to_run_lcdm) > 0:
    print(f'  ({", ".join(need_to_run_lcdm)})')
    print(f'Estimated compute: ~{len(need_to_run_lcdm) * 30} minutes')


Checking ΛCDM v3 pickle files:
  ✓ iso_v3_akarsu_P_noCC_lcdm: already at standard name
  ✓ iso_v3_akarsu_P_CC_lcdm: already at standard name
  ✓ iso_v3_akarsu_PP_noCC_lcdm: already at standard name
  ✓ iso_v3_akarsu_PP_CC_lcdm: already at standard name
  ✓ iso_v3_dr2_P_noCC_lcdm: already at standard name
  ✓ iso_v3_dr2_P_CC_lcdm: already at standard name
  ✓ iso_v3_dr2_PP_noCC_lcdm: already at standard name
  ✓ iso_v3_dr2_PP_CC_lcdm: already at standard name
  ✓ iso_v3_baotr_P_noCC_lcdm: already at standard name
  ✓ iso_v3_baotr_P_CC_lcdm: already at standard name
  ✓ iso_v3_baotr_PP_noCC_lcdm: already at standard name
  ✓ iso_v3_baotr_PP_CC_lcdm: already at standard name
Reused ΛCDM: 12/12 fits
To run ΛCDM: 0 fits


In [25]:
results_lcdm = {}
for name in ISO_NAMES_LCDM:
    r = run_or_load_iso_lcdm(name)
    results_lcdm[name] = r

print()
print('=' * 70)
print('  ALL 12 ΛCDM ISOLATION FITS COMPLETE')
print('=' * 70)


  [iso_v3_akarsu_P_noCC_lcdm] loaded from cache
  [iso_v3_akarsu_P_CC_lcdm] loaded from cache
  [iso_v3_akarsu_PP_noCC_lcdm] loaded from cache
  [iso_v3_akarsu_PP_CC_lcdm] loaded from cache
  [iso_v3_dr2_P_noCC_lcdm] loaded from cache
  [iso_v3_dr2_P_CC_lcdm] loaded from cache
  [iso_v3_dr2_PP_noCC_lcdm] loaded from cache
  [iso_v3_dr2_PP_CC_lcdm] loaded from cache
  [iso_v3_baotr_P_noCC_lcdm] loaded from cache
  [iso_v3_baotr_P_CC_lcdm] loaded from cache
  [iso_v3_baotr_PP_noCC_lcdm] loaded from cache
  [iso_v3_baotr_PP_CC_lcdm] loaded from cache

  ALL 12 ΛCDM ISOLATION FITS COMPLETE


## 23. Bayes Factor Analysis

The **physically meaningful quantity** is the Bayes factor:

$$\Delta\ln Z \equiv \ln Z(\Lambda_s\text{CDM}) - \ln Z(\Lambda\text{CDM})$$

**Sign convention** (matches Akarsu et al. 2022):
- $\Delta\ln Z > 0$: Λ$_s$CDM preferred over ΛCDM
- $\Delta\ln Z < 0$: ΛCDM preferred over Λ$_s$CDM

**Jeffreys' scale**:
- $|\Delta\ln Z| < 1$: inconclusive
- $1 < |\Delta\ln Z| < 2.5$: weak
- $2.5 < |\Delta\ln Z| < 5$: moderate
- $|\Delta\ln Z| > 5$: strong/decisive


In [26]:
def get_bf(bao, snia, cc):
    """Compute Bayes factor ΔlnZ = lnZ(LsCDM) - lnZ(ΛCDM) for a combo."""
    name_lscdm = f'iso_v3_{bao}_{snia}_{cc}'
    name_lcdm = f'iso_v3_{bao}_{snia}_{cc}_lcdm'
    lnZ_lscdm = results[name_lscdm]['log_z']
    lnZ_lcdm = results_lcdm[name_lcdm]['log_z']
    err_lscdm = results[name_lscdm]['log_z_err']
    err_lcdm = results_lcdm[name_lcdm]['log_z_err']
    bf = lnZ_lscdm - lnZ_lcdm
    bf_err = np.sqrt(err_lscdm**2 + err_lcdm**2)
    return bf, bf_err


print('=' * 90)
print('  BAYES FACTOR TABLE — ΔlnZ = lnZ(Λ_sCDM) - lnZ(ΛCDM)')
print('=' * 90)
print()
print(f'  {"BAO":<8} {"SnIa":<6} {"CC":<6} '
      f'{"lnZ(LsCDM)":<14} {"lnZ(LCDM)":<14} {"ΔlnZ":<14} {"Verdict"}')
print('-' * 90)

bao_order = ['akarsu', 'dr2', 'baotr']
ordered_combos = [
    (b, s, c)
    for b in bao_order for s in ['P', 'PP'] for c in ['noCC', 'CC']
]

for bao, snia, cc in ordered_combos:
    name_lscdm = f'iso_v3_{bao}_{snia}_{cc}'
    name_lcdm = f'iso_v3_{bao}_{snia}_{cc}_lcdm'
    lnZ_lscdm = results[name_lscdm]['log_z']
    lnZ_lcdm = results_lcdm[name_lcdm]['log_z']
    bf, bf_err = get_bf(bao, snia, cc)

    abs_bf = abs(bf)
    if abs_bf < 1:
        verdict = 'inconclusive'
    elif abs_bf < 2.5:
        verdict = 'weak'
    elif abs_bf < 5:
        verdict = 'moderate'
    else:
        verdict = 'STRONG'

    direction = ' (LsCDM)' if bf > 0 else ' (ΛCDM)'

    print(f'  {bao:<8} {snia:<6} {cc:<6} '
          f'{lnZ_lscdm:>10.2f}     {lnZ_lcdm:>10.2f}     '
          f'{bf:+8.2f} ± {bf_err:.2f}  {verdict}{direction}')

print('=' * 90)
print()
print('Sign convention: ΔlnZ > 0 means Λ_sCDM preferred over ΛCDM')
print('                 ΔlnZ < 0 means ΛCDM preferred over Λ_sCDM')


  BAYES FACTOR TABLE — ΔlnZ = lnZ(Λ_sCDM) - lnZ(ΛCDM)

  BAO      SnIa   CC     lnZ(LsCDM)     lnZ(LCDM)      ΔlnZ           Verdict
------------------------------------------------------------------------------------------
  akarsu   P      noCC      -543.68        -546.29        +2.61 ± 0.12  moderate (LsCDM)
  akarsu   P      CC        -550.98        -553.72        +2.73 ± 0.12  moderate (LsCDM)
  akarsu   PP     noCC      -761.81        -764.18        +2.37 ± 0.12  weak (LsCDM)
  akarsu   PP     CC        -769.08        -771.61        +2.53 ± 0.12  moderate (LsCDM)
  dr2      P      noCC      -543.75        -546.19        +2.44 ± 0.13  weak (LsCDM)
  dr2      P      CC        -551.06        -553.59        +2.53 ± 0.13  moderate (LsCDM)
  dr2      PP     noCC      -761.89        -764.56        +2.67 ± 0.13  moderate (LsCDM)
  dr2      PP     CC        -769.20        -771.97        +2.77 ± 0.13  moderate (LsCDM)
  baotr    P      noCC      -540.76        -556.70       +15.94 ± 0.12  

## 24. Marginal Effects on Bayes Factor

For each axis, compute the marginal effect on the Bayes factor by
averaging $\Delta\Delta\ln Z$ over the other two axes.

This is the **correct quantity**: it cancels the dataset-size effect
that contaminates raw $\ln Z$ comparisons.

**Sign convention**: positive marginal = upgrade increases preference for
Λ$_s$CDM; negative = upgrade decreases preference.


In [27]:
def get_bf_only(bao, snia, cc):
    bf, _ = get_bf(bao, snia, cc)
    return bf


# BAO axis: Akarsu → DR2 (averaged over SnIa, CC)
delta_bf_akarsu_to_dr2 = np.mean([
    get_bf_only('dr2', s, c) - get_bf_only('akarsu', s, c)
    for s in ['P', 'PP'] for c in ['noCC', 'CC']
])

# BAO axis: Akarsu → BAOtr
delta_bf_akarsu_to_baotr = np.mean([
    get_bf_only('baotr', s, c) - get_bf_only('akarsu', s, c)
    for s in ['P', 'PP'] for c in ['noCC', 'CC']
])

# SnIa axis: P → PP
delta_bf_p_to_pp = np.mean([
    get_bf_only(b, 'PP', c) - get_bf_only(b, 'P', c)
    for b in bao_order for c in ['noCC', 'CC']
])

# CC axis: noCC → CC
delta_bf_noCC_to_CC = np.mean([
    get_bf_only(b, s, 'CC') - get_bf_only(b, s, 'noCC')
    for b in bao_order for s in ['P', 'PP']
])


print('=' * 80)
print('  MARGINAL EFFECTS ON BAYES FACTOR — ΔΔlnZ')
print('=' * 80)
print()
print('Sign convention: positive = upgrade favors Λ_sCDM')
print('                 negative = upgrade disfavors Λ_sCDM')
print()
print(f'  BAO axis: Akarsu → DR2:     ΔΔlnZ = {delta_bf_akarsu_to_dr2:+.3f}')
print(f'  BAO axis: Akarsu → BAOtr:   ΔΔlnZ = {delta_bf_akarsu_to_baotr:+.3f}')
print(f'  SnIa axis: Pantheon → PP:   ΔΔlnZ = {delta_bf_p_to_pp:+.3f}')
print(f'  CC axis: noCC → Moresco:    ΔΔlnZ = {delta_bf_noCC_to_CC:+.3f}')
print()
print('=' * 80)
print()

# Interpretation (positive = favors Λ_sCDM)
print('Interpretation:')
# BAO: Akarsu (SDSS-era 3D) → DESI DR2 (3D anisotropic)
if abs(delta_bf_akarsu_to_dr2) < 1.0:
    print(f'  ✓ Akarsu → DR2 is NEGLIGIBLE ({delta_bf_akarsu_to_dr2:+.2f}): the two 3D')
    print('    anisotropic compilations give essentially the same preference.')
elif delta_bf_akarsu_to_dr2 > 0:
    print(f'  → DR2 FAVORS Λ_sCDM by +{delta_bf_akarsu_to_dr2:.2f} ln Z relative to Akarsu')
else:
    print(f'  → DR2 DISFAVORS Λ_sCDM by {abs(delta_bf_akarsu_to_dr2):.2f} ln Z relative to Akarsu')
# BAO: Akarsu (3D) → BAOtr (transversal / 2D)
if delta_bf_akarsu_to_baotr > 1.0:
    print(f'  ✓ BAOtr FAVORS Λ_sCDM by +{delta_bf_akarsu_to_baotr:.1f} ln Z')
    print('    → preference tracks BAO TYPE (transversal/2D vs 3D), not Akarsu→DESI.')
elif abs(delta_bf_akarsu_to_baotr) < 1.0:
    print(f'  → BAOtr is approximately neutral ({delta_bf_akarsu_to_baotr:+.2f})')
else:
    print(f'  → BAOtr DISFAVORS Λ_sCDM by {abs(delta_bf_akarsu_to_baotr):.1f} ln Z')
if abs(delta_bf_p_to_pp) < 1.0:
    print(f'  ✓ SnIa upgrade (Pantheon → Pantheon+) is NEGLIGIBLE ({delta_bf_p_to_pp:+.2f})')
if abs(delta_bf_noCC_to_CC) < 1.0:
    print(f'  ✓ CC inclusion is NEGLIGIBLE ({delta_bf_noCC_to_CC:+.2f})')

  MARGINAL EFFECTS ON BAYES FACTOR — ΔΔlnZ

Sign convention: positive = upgrade favors Λ_sCDM
                 negative = upgrade disfavors Λ_sCDM

  BAO axis: Akarsu → DR2:     ΔΔlnZ = +0.042
  BAO axis: Akarsu → BAOtr:   ΔΔlnZ = +12.286
  SnIa axis: Pantheon → PP:   ΔΔlnZ = -0.685
  CC axis: noCC → Moresco:    ΔΔlnZ = +0.046


Interpretation:
  ✓ Akarsu → DR2 is NEGLIGIBLE (+0.04): the two 3D
    anisotropic compilations give essentially the same preference.
  ✓ BAOtr FAVORS Λ_sCDM by +12.3 ln Z
    → preference tracks BAO TYPE (transversal/2D vs 3D), not Akarsu→DESI.
  ✓ SnIa upgrade (Pantheon → Pantheon+) is NEGLIGIBLE (-0.68)
  ✓ CC inclusion is NEGLIGIBLE (+0.05)


## 25. Final LaTeX Table for Paper (with ΔBF)


In [28]:
header = r"""\begin{table}
\caption{$2\times 2\times 2 \times 3$ isolation analysis with Bayes factors.
For each combination of BAO compilation, SnIa sample, and Cosmic
Chronometer choice, we report $\ln Z$ for both $\Lambda_s$CDM and $\Lambda$CDM,
and the Bayes factor $\Delta\ln Z = \ln Z(\Lambda_s\mathrm{CDM}) - \ln Z(\Lambda\mathrm{CDM})$.
The $M_B$ prior from SH0ES is applied throughout. The $z_\dagger$ prior is $[1, 3]$.
$\Delta\ln Z > 0$ favors $\Lambda_s$CDM; $\Delta\ln Z < 0$ favors $\Lambda$CDM.}
\centering
\begin{tabular}{lll|ccc}
\toprule
BAO & SnIa & CC & $\ln Z(\Lambda_s\mathrm{CDM})$ & $\ln Z(\Lambda\mathrm{CDM})$ & $\Delta\ln Z$ \\
\midrule"""

footer = r"""\bottomrule
\end{tabular}
\end{table}"""

print(header)

bao_label = {'akarsu': 'Akarsu (SDSS)', 'dr2': 'DESI DR2', 'baotr': 'BAOtr'}
snia_label = {'P': 'Pantheon', 'PP': 'Pantheon+'}
cc_label = {'noCC': 'none', 'CC': 'Moresco 2022'}

for bao, snia, cc in ordered_combos:
    name_lscdm = f'iso_v3_{bao}_{snia}_{cc}'
    name_lcdm = f'iso_v3_{bao}_{snia}_{cc}_lcdm'
    lnZ_lscdm = results[name_lscdm]['log_z']
    lnZ_lcdm = results_lcdm[name_lcdm]['log_z']
    bf, bf_err = get_bf(bao, snia, cc)

    line = (f'{bao_label[bao]} & {snia_label[snia]} & {cc_label[cc]} '
            f'& ${lnZ_lscdm:.2f}$ & ${lnZ_lcdm:.2f}$ '
            f'& ${bf:+.2f} \\pm {bf_err:.2f}$ \\\\')
    print(line)

print(footer)
print()
print()

# Marginal effects table
print(r'\begin{table}')
print(r'\caption{Marginal effects on the Bayes factor $\Delta\ln Z$, '
      r'averaged over the other axes.}')
print(r'\centering')
print(r'\begin{tabular}{l|c}')
print(r'\toprule')
print(r'Axis upgrade & $\Delta\Delta\ln Z$ \\')
print(r'\midrule')
print(rf'BAO: Akarsu $\to$ DESI DR2 & {delta_bf_akarsu_to_dr2:+.3f} \\')
print(rf'BAO: Akarsu $\to$ BAOtr & {delta_bf_akarsu_to_baotr:+.3f} \\')
print(rf'SnIa: Pantheon $\to$ Pantheon+ & {delta_bf_p_to_pp:+.3f} \\')
print(rf'CC: none $\to$ Moresco 2022 & {delta_bf_noCC_to_CC:+.3f} \\')
print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')


\begin{table}
\caption{$2\times 2\times 2 \times 3$ isolation analysis with Bayes factors.
For each combination of BAO compilation, SnIa sample, and Cosmic
Chronometer choice, we report $\ln Z$ for both $\Lambda_s$CDM and $\Lambda$CDM,
and the Bayes factor $\Delta\ln Z = \ln Z(\Lambda_s\mathrm{CDM}) - \ln Z(\Lambda\mathrm{CDM})$.
The $M_B$ prior from SH0ES is applied throughout. The $z_\dagger$ prior is $[1, 3]$.
$\Delta\ln Z > 0$ favors $\Lambda_s$CDM; $\Delta\ln Z < 0$ favors $\Lambda$CDM.}
\centering
\begin{tabular}{lll|ccc}
\toprule
BAO & SnIa & CC & $\ln Z(\Lambda_s\mathrm{CDM})$ & $\ln Z(\Lambda\mathrm{CDM})$ & $\Delta\ln Z$ \\
\midrule
Akarsu (SDSS) & Pantheon & none & $-543.68$ & $-546.29$ & $+2.61 \pm 0.12$ \\
Akarsu (SDSS) & Pantheon & Moresco 2022 & $-550.98$ & $-553.72$ & $+2.73 \pm 0.12$ \\
Akarsu (SDSS) & Pantheon+ & none & $-761.81$ & $-764.18$ & $+2.37 \pm 0.12$ \\
Akarsu (SDSS) & Pantheon+ & Moresco 2022 & $-769.08$ & $-771.61$ & $+2.53 \pm 0.12$ \\
DESI DR2 & Pantheon

## 25. Wide-prior [1,5] fits (PP) for Part A §6.4 right column

In [29]:
# ================================================================
# §25. Wide-prior (z† ∈ [1,5]) fits for the §6.4 right-hand column.
#      Saves iso_v3_<bao>_PP_CC_zt5.pkl (PP only).
#      Re-runs the [1,5] versions of the PP CC fits so the §6.4
#      right-hand column can be populated.
# ================================================================
import pickle

_ZT_MAX_SAVED = ZT_MAX          # remember the baseline
ZT_MAX = 5.0                    # widen the z† prior for these runs

_zt5_snia = ['PP']

try:
    for snia in _zt5_snia:
        for bao in ['akarsu', 'dr2', 'baotr']:
            name = f'iso_v3_{bao}_{snia}_CC_zt5'
            if os.path.exists(f'{name}.pkl'):
                print(f'  ✓ {name}.pkl exists — skipping'); continue
            print(f'  running {name} ...')
            prior = make_prior_lsCDM()         # picks up ZT_MAX = 5.0
            def loglike(p, bao=bao, snia=snia):
                return -0.5*chi2_isolation(bao, snia, 'CC',
                                           p['om'], p['h'], p['obh2'], p['zt'], p['M'])
            smp = Sampler(prior, loglike, n_live=1500)
            smp.run(verbose=False)
            pts, lw, ll = smp.posterior()
            w = np.exp(lw - lw.max()); w /= w.sum()
            logZ, dlogZ = logZ_with_err(smp)
            with open(f'{name}.pkl', 'wb') as f:
                pickle.dump({'name': name, 'points': pts, 'weights': w,
                             'log_like': ll, 'log_z': logZ, 'log_z_err': dlogZ,
                             'chi2_min': -2*ll.max(),
                             'config': {'bao': bao, 'snia': snia, 'cc': 'CC', 'zt_max': 5.0}}, f)
            zt = pts[:, 3]; order = np.argsort(zt); cdf = np.cumsum(w[order])
            lo, md_, hi = np.interp([0.16, 0.5, 0.84], cdf, zt[order])
            frac = float(w[(zt >= 1.5) & (zt <= 2.6)].sum())
            print(f'    z† = {md_:.3f} +{hi-md_:.3f}/-{md_-lo:.3f},  {frac*100:.0f}% in [1.5,2.6]')
finally:
    ZT_MAX = _ZT_MAX_SAVED      # restore baseline no matter what

print('\nWide-prior fits complete. Part A §6.4 will fill the [1,5] column automatically.')

  ✓ iso_v3_akarsu_PP_CC_zt5.pkl exists — skipping
  ✓ iso_v3_dr2_PP_CC_zt5.pkl exists — skipping
  ✓ iso_v3_baotr_PP_CC_zt5.pkl exists — skipping

Wide-prior fits complete. Part A §6.4 will fill the [1,5] column automatically.
